Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/lucianommartins/lab-sabadao/blob/main/examples/notebooks/01_gbench_101_quickstart.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench 101: Installation, Ollama setup, and quick performance check

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This interactive notebook introduces `gbench`, an open-source performance benchmarking and capability evaluation suite for foundation models. You will learn how to install the package, configure a local Ollama serving engine with a quantized Google Gemma model, execute a baseline performance check, and inspect the resulting metrics.

## Learning objectives

1. Install `gbench` and its required dependencies in an isolated Python environment.
2. Configure and start a local Ollama server running a quantized Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
3. Run a quick performance smoke test using `gbench --preset quick` over standard OpenAI `/v1` REST endpoints.
4. Load and inspect JSON benchmark outputs using Pandas to analyze Time to First Token (TTFT), Time per Output Token (TPOT), and request throughput.
5. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [lab-sabadao GitHub repository](https://github.com/lucianommartins/lab-sabadao)
* [Ollama documentation](https://github.com/ollama/ollama)
* [Unsloth Gemma 4 QAT GGUF checkpoints](https://huggingface.co/unsloth/gemma-4-E4B-it-qat-GGUF)

## 1. Environment setup and installation

We clone the `lab-sabadao` repository from GitHub, change directory into the project root (`%cd lab-sabadao`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os
from pathlib import Path

# Safe environment setup (prevents recursive nesting on re-run)
if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
    if not Path("lab-sabadao").is_dir():
        !git clone https://github.com/lucianommartins/lab-sabadao.git
    %cd lab-sabadao

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available presets
!gbench --list presets

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil

if not shutil.which("ollama"):
    print("Installing Ollama locally...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama binary already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Writing custom Modelfile for QAT model

We create an Ollama `Modelfile.qat` that configures our quantized Google Gemma 4 model (`hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:latest`) with explicit parameters:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

In [ ]:
HF_MODEL_ID = "hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:UD-Q4_K_XL"
modelfile_content = f"""FROM {HF_MODEL_ID}
PARAMETER num_ctx 8192
SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."
"""
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write(modelfile_content)
print("Created Modelfile.qat with valid Ollama parameters (num_ctx 8192, SYSTEM prompt).")

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) using `ollama create -f Modelfile.qat`. This pulls the GGUF weights from Hugging Face Hub if not already cached. We then run a quick generation test (`ollama run`) to verify that the model loads into hardware memory and generates tokens correctly.

In [ ]:
MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from Modelfile.qat...")
!ollama create {MODEL_TAG} -f Modelfile.qat
print("Running quick generation smoke test...")
!ollama run {MODEL_TAG} "Reply with the single word: READY."

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in resp.json().get("data", [])]
print("Available REST models:", models)

## 7. Running a quick performance smoke test

With Ollama running locally, we execute a quick performance smoke test using `gbench --preset quick`. This preset runs 1 warmup iteration and 1 test iteration at concurrency `batch_size=1`, providing a rapid baseline check of prefill latency and decode speed.

By passing `--remote-endpoint http://localhost:11434/v1`, we instruct `gbench` to communicate directly with Ollama's OpenAI `/v1` REST interface without checking local GPU VRAM or attempting to spawn a vLLM subprocess.

In [ ]:
!gbench --models gemma4-qat:4b \
        --preset quick \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_101 \
        --num-prompts 10

## 8. Inspecting benchmark results in Python

Every `gbench` run outputs JSON summaries and sample traces into a timestamped directory under `--results-dir`. We can load `summary.json` into a Pandas DataFrame to inspect core serving metrics:
* **`request_throughput`**: Completed HTTP requests per second.
* **`output_token_throughput`**: Generated tokens per second across all concurrent streams.
* **`ttft_ms`**: Time to First Token prefill latency (mean, median, P99).
* **`tpot_ms`**: Time per Output Token decoding latency.

In [ ]:
import json, glob, os
from pathlib import Path
import pandas as pd

# Find the latest result run folder
results_base = Path("./results_101")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True) if results_base.exists() else []

if run_dirs:
    latest_dir = run_dirs[0]
    summary_path = latest_dir / "summary.json"
    
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data.get("models", []))
    else:
        # Fallback: load individual performance run JSON files
        perf_files = sorted(latest_dir.glob("performance/serve_*.json"))
        records = []
        for pf in perf_files:
            with open(pf, "r", encoding="utf-8") as f:
                records.append(json.load(f))
        df = pd.DataFrame(records)

    if not df.empty:
        cols = [c for c in ["model", "format", "batch_size", "request_throughput", "output_token_throughput", "mean_ttft_ms", "median_ttft_ms", "ttft_p50_ms", "mean_tpot_ms", "tpot_p50_ms"] if c in df.columns]
        display(df[cols])
    else:
        print("No benchmark records found in:", latest_dir)
else:
    print("No benchmark result directory found.")

## 9. Session cleanup and server shutdown

To prevent VRAM fragmentation and orphan background processes, we cleanly shut down the local Ollama serving engine at the end of every notebook session.

In [ ]:
import subprocess, os

print("Stopping Ollama server processes...")
subprocess.run(["pkill", "-f", "ollama"], check=False)

if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")

print("Session cleanup complete. VRAM and system memory reclaimed.")